In [1]:
import torch
import torch.nn as nn
import timm
import pandas as pd
import numpy as np
import os
import math
import json
from PIL import Image
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

<jemalloc>: Unsupported system page size


Device: cuda:3


In [2]:
data_dir = "/datasets/multi-view-pig-posture-recognition/"
train2_df = pd.read_csv(os.path.join(data_dir, "train2.csv"))
dir_train2 = os.path.join(data_dir, "train2_images")

test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))
dir_test = os.path.join(data_dir, "test_images")

def parse_bbox_safe(x):
    if isinstance(x, str):
        return [float(i) for i in x.replace("[", "").replace("]", "").split(",")]
    return x

train2_df["bbox"] = train2_df["bbox"].apply(parse_bbox_safe)
test_df["bbox"] = test_df["bbox"].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

train2_df["camera_id"] = train2_df["image_id"].apply(lambda x: x.split("_")[0])

print(f"Train2: {len(train2_df)} samples")
print(f"Test:   {len(test_df)} samples")
print(f"\nKlassenverteilung:")
print(train2_df["class_id"].value_counts().sort_index())
print(f"\nKameras: {train2_df['camera_id'].nunique()}")

Train2: 23450 samples
Test:   11708 samples

Klassenverteilung:
class_id
0    3083
1    3435
2     695
3    9928
4    6309
Name: count, dtype: int64

Kameras: 2


In [3]:
class PigPostureDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"])
        image = Image.open(img_path).convert("RGB")

        bbox = row["bbox"]
        x, y, w, h = bbox
        # Expand bbox by 15% for context
        cx, cy = x + w / 2, y + h / 2
        w_exp, h_exp = w * 1.15, h * 1.15
        x_exp, y_exp = cx - w_exp / 2, cy - h_exp / 2

        img_w, img_h = image.size
        x_min = max(0, int(x_exp))
        y_min = max(0, int(y_exp))
        x_max = min(img_w, int(x_exp + w_exp))
        y_max = min(img_h, int(y_exp + h_exp))
        if x_max > x_min and y_max > y_min:
            image = image.crop((x_min, y_min, x_max, y_max))

        if self.transform:
            image = self.transform(image)

        label = int(row["class_id"])
        return image, label

print("✅ PigPostureDataset ready (with 15% bbox expansion)")

✅ PigPostureDataset ready (with 15% bbox expansion)


In [4]:
IMG_SIZE = 448  # Native resolution for EVA-02 Large

train_df, val_df = train_test_split(
    train2_df, test_size=0.2, stratify=train2_df["class_id"], random_state=42
)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = PigPostureDataset(train_df, dir_train2, transform=train_transform)
val_dataset = PigPostureDataset(val_df, dir_train2, transform=val_transform)

# Class weights with sqrt dampening
train_labels = train_df["class_id"].tolist()
class_counts = Counter(train_labels)
num_classes = len(class_counts)
total = sum(class_counts.values())

class_weights = []
for i in range(num_classes):
    w = math.sqrt(total / (num_classes * class_counts[i]))
    class_weights.append(w)
    print(f"Klasse {i}: {class_counts[i]} samples, weight={w:.3f}")

class_weights_tensor = torch.FloatTensor(class_weights).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

BATCH_SIZE = 16  # Smaller batch for larger model

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)

print(f"\n✅ Train: {len(train_loader)} Batches, Val: {len(val_loader)} Batches")

Train: 18760, Val: 4690
Klasse 0: 2466 samples, weight=1.233
Klasse 1: 2748 samples, weight=1.168
Klasse 2: 556 samples, weight=2.598
Klasse 3: 7943 samples, weight=0.687
Klasse 4: 5047 samples, weight=0.862

✅ Train: 1172 Batches, Val: 294 Batches


In [5]:
class PigPostureModel(nn.Module):
    def __init__(self, num_classes=5, backbone_name="eva02_large_patch14_448.mim_m38m_ft_in22k_in1k"):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.backbone.requires_grad_(False)
        feat_dim = self.backbone.num_features

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(0.4),
            nn.Linear(feat_dim, 768),
            nn.GELU(),
            nn.LayerNorm(768),
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    def unfreeze_backbone(self, fraction=0.5):
        params = list(self.backbone.parameters())
        n_unfreeze = int(len(params) * fraction)
        for p in params[-n_unfreeze:]:
            p.requires_grad = True
        print(f"Unfroze {n_unfreeze}/{len(params)} backbone params")

model = PigPostureModel(num_classes=num_classes)
model.to(device)
print(f"✅ EVA-02 Large Model ready, features: {model.backbone.num_features}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

✅ EVA-02 Large Model ready, features: 1024


In [8]:
class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, max_epochs, min_lr_ratio=0.01):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.min_lr_ratio = min_lr_ratio
        self.current_epoch = 0

    def step(self):
        self.current_epoch += 1
        if self.current_epoch <= self.warmup_epochs:
            lr_scale = self.current_epoch / self.warmup_epochs
        else:
            progress = (self.current_epoch - self.warmup_epochs) / (self.max_epochs - self.warmup_epochs)
            lr_scale = self.min_lr_ratio + 0.5 * (1 - self.min_lr_ratio) * (1 + math.cos(math.pi * progress))
        for pg in self.optimizer.param_groups:
            pg['lr'] = pg['initial_lr'] * lr_scale

    def get_last_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]

def setup_scheduler(optimizer, warmup, max_ep):
    for pg in optimizer.param_groups:
        pg['initial_lr'] = pg['lr']
    return CosineWarmupScheduler(optimizer, warmup, max_ep)

def mixup_data(x, y, alpha=0.3):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ✅ Kompatible GradScaler- und autocast-Auswahl je nach PyTorch-Version
if hasattr(torch.amp, "GradScaler"):
    # PyTorch >= 2.1
    def make_scaler():
        return torch.amp.GradScaler("cuda")
    def autocast_ctx():
        return torch.amp.autocast("cuda")
else:
    # PyTorch < 2.1
    def make_scaler():
        return torch.cuda.amp.GradScaler()
    def autocast_ctx():
        return torch.cuda.amp.autocast()

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs=10, save_name="best_model.pth", accumulation_steps=4, use_mixup=True):
    best_acc = 0.0
    scaler = make_scaler()  # ✅ versionssicher

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()

        for i, (images, labels) in enumerate(tqdm(train_loader, desc=f"Ep {epoch+1}/{num_epochs} Train")):
            images, labels = images.to(device), labels.to(device)

            if use_mixup and np.random.rand() < 0.5:
                images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.3)
                with autocast_ctx():  # ✅ versionssicher
                    outputs = model(images)
                    loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam) / accumulation_steps
            else:
                with autocast_ctx():  # ✅ versionssicher
                    outputs = model(images)
                    loss = criterion(outputs, labels) / accumulation_steps

            scaler.scale(loss).backward()

            if (i + 1) % accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            running_loss += loss.item() * accumulation_steps

        model.eval()
        correct, total_val = 0, 0
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Ep {epoch+1}/{num_epochs} Val"):
                images, labels = images.to(device), labels.to(device)
                with autocast_ctx():  # ✅ versionssicher
                    outputs = model(images)
                    val_loss += criterion(outputs, labels).item()
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        val_loss = val_loss / len(val_loader)
        val_acc = correct / total_val
        lr = scheduler.get_last_lr()[0]
        scheduler.step()

        saved = ""
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_name)
            saved = " ✅ SAVED"

        print(f"Ep {epoch+1} | TrL: {train_loss:.4f} | VaL: {val_loss:.4f} | "
              f"Acc: {val_acc:.4f} | LR: {lr:.2e}{saved}")
    return best_acc

print(f"✅ Training function ready (PyTorch {torch.__version__}, Mixup + AMP)")

✅ Training function ready (PyTorch 1.12.1, Mixup + AMP)


In [ ]:
optimizer = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = setup_scheduler(optimizer, warmup=2, max_ep=8)

print("=== PHASE 1: Train Head (backbone frozen) ===")
best_acc = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                       num_epochs=8, save_name="best_eva02_phase1.pth",
                       accumulation_steps=4, use_mixup=False)
print(f"\n🏆 Phase 1 Best Acc: {best_acc:.4f}")

=== PHASE 1: Train Head (backbone frozen) ===


Ep 1/8 Train:   5%|▌         | 63/1172 [00:39<10:50,  1.70it/s] 

In [ ]:
model.load_state_dict(torch.load("best_eva02_phase1.pth", map_location=device))
model.unfreeze_backbone(fraction=0.3)

optimizer_ft1 = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": 2e-6},
    {"params": model.head.parameters(), "lr": 2e-5},
], weight_decay=1e-2)
scheduler_ft1 = setup_scheduler(optimizer_ft1, warmup=2, max_ep=8)

print("=== PHASE 2: Fine-Tune 30% Backbone ===")
best_acc = train_model(model, train_loader, val_loader, criterion, optimizer_ft1, scheduler_ft1,
                       num_epochs=8, save_name="best_eva02_phase2.pth",
                       accumulation_steps=4, use_mixup=True)
print(f"\n🏆 Phase 2 Best Acc: {best_acc:.4f}")

In [ ]:
model.load_state_dict(torch.load("best_eva02_phase2.pth", map_location=device))
# Unfreeze more
model.unfreeze_backbone(fraction=0.6)

optimizer_ft2 = torch.optim.AdamW([
    {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": 5e-7},
    {"params": model.head.parameters(), "lr": 5e-6},
], weight_decay=1e-2)
scheduler_ft2 = setup_scheduler(optimizer_ft2, warmup=1, max_ep=5)

print("=== PHASE 3: Fine-Tune 60% Backbone (low LR) ===")
best_acc = train_model(model, train_loader, val_loader, criterion, optimizer_ft2, scheduler_ft2,
                       num_epochs=5, save_name="best_eva02_phase3.pth",
                       accumulation_steps=4, use_mixup=True)
print(f"\n🏆 Phase 3 Best Acc: {best_acc:.4f}")

In [ ]:
best_model_path = "best_eva02_phase3.pth"
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)
model.eval()

# TTA transforms
tta_transforms = [
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]),
]

class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"])
        image = Image.open(img_path).convert("RGB")

        bbox = row["bbox"]
        if isinstance(bbox, str):
            bbox = json.loads(bbox)
        x, y, w, h = bbox
        # Same 15% expansion as training
        cx, cy = x + w / 2, y + h / 2
        w_exp, h_exp = w * 1.15, h * 1.15
        x_exp, y_exp = cx - w_exp / 2, cy - h_exp / 2

        img_w, img_h = image.size
        x_min = max(0, int(x_exp))
        y_min = max(0, int(y_exp))
        x_max = min(img_w, int(x_exp + w_exp))
        y_max = min(img_h, int(y_exp + h_exp))
        if x_max > x_min and y_max > y_min:
            image = image.crop((x_min, y_min, x_max, y_max))

        image = self.transform(image)
        return image, row["row_id"]

# TTA inference: average logits across augmentations
all_logits = {}

for tta_idx, tta_tf in enumerate(tta_transforms):
    print(f"\nTTA {tta_idx + 1}/{len(tta_transforms)}...")
    test_dataset = TestDataset(test_df, dir_test, tta_tf)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

    with torch.no_grad():
        for images, rids in tqdm(test_loader, desc=f"TTA {tta_idx+1}"):
            images = images.to(device)
            with torch.amp.autocast("cuda"):
                outputs = model(images)
            outputs = outputs.float().cpu().numpy()
            for j, rid in enumerate(rids):
                rid_str = str(rid)
                if rid_str not in all_logits:
                    all_logits[rid_str] = outputs[j]
                else:
                    all_logits[rid_str] += outputs[j]

# Final predictions
row_ids = []
predictions = []
for rid in all_logits:
    row_ids.append(rid)
    predictions.append(int(np.argmax(all_logits[rid])))

submission = pd.DataFrame({"row_id": row_ids, "class_id": predictions})
submission.to_csv("eva02_large_tta_submission.csv", index=False)

print(f"\n✅ Gespeichert als: eva02_large_tta_submission.csv")
print(f"📊 Modell: {best_model_path}")
print(f"📊 TTA: {len(tta_transforms)} augmentations")
print(f"\nKlassenverteilung:")
print(submission["class_id"].value_counts().sort_index())
print(submission.head())